In [ ]:
#|default_exp postprocess










# Post-Processing of Refined Data

Deterministic post-processing fixes applied after LLM refinement.
Addresses known issues: nan task types, LLM artifacts in text,
US-specific terminology, wrong-domain tasks, and essential tech
skills wrongly removed.

In [ ]:
#|export
import logging
import re

from aspectt_pipeline.const import VALID_TASK_TYPES

logger = logging.getLogger(__name__)










## 1.1 Normalise task_type values

The LLM refinement step now constrains task_type to a Literal enum,
but we still normalise as a safety net for any values that slip through
(e.g. from unrefined data or cached old LLM responses).

In [ ]:
#|export
def _normalise_task_types(occ: dict) -> int:
    """Normalise task_type to one of Core/Supplemental/Unclassified. Returns count of fixes."""
    fixes = 0
    for task in occ.get('tasks', []):
        tt = task.get('task_type')
        if tt not in VALID_TASK_TYPES:
            # Try case-insensitive match
            if isinstance(tt, str):
                for valid in VALID_TASK_TYPES:
                    if tt.lower() == valid.lower():
                        task['task_type'] = valid
                        fixes += 1
                        break
                else:
                    task['task_type'] = 'Unclassified'
                    fixes += 1
            else:
                task['task_type'] = 'Unclassified'
                fixes += 1
    return fixes










## 1.2 Remove empty or placeholder tasks

Safety net: remove tasks with empty text or obvious placeholder content.
With the Literal-constrained task_type and improved prompts, most LLM
artifacts should no longer occur, but we still clean up any stragglers.

In [ ]:
#|export
_PLACEHOLDER_PATTERNS = [
    re.compile(r'^Remove (irrelevant |tasks)', re.IGNORECASE),
    re.compile(r'\[REMOVED\]', re.IGNORECASE),
    re.compile(r'\[REDACTED\]', re.IGNORECASE),
    re.compile(r'^Not relevant\.?$', re.IGNORECASE),
]


def _remove_placeholder_tasks(occ: dict) -> int:
    """Remove tasks with empty or placeholder text. Returns count removed."""
    tasks = occ.get('tasks', [])
    if not tasks:
        return 0

    clean = []
    removed = 0
    for task in tasks:
        text = task.get('task', '').strip()

        # Remove empty tasks
        if not text:
            removed += 1
            continue

        # Remove placeholder/artifact text
        if any(p.search(text) for p in _PLACEHOLDER_PATTERNS):
            removed += 1
            continue

        clean.append(task)

    occ['tasks'] = clean
    return removed










## 1.3 Generic Tech Skill Whitelist

The LLM occasionally removes ubiquitous technologies (Microsoft Office, email,
web browsers) that are genuinely relevant to almost all occupations. As a
safety net, we check the unrefined data for these whitelisted items and
re-add any that were wrongly removed.

In [ ]:
#|export
GENERIC_TECH_WHITELIST = {
    'Microsoft Office',
    'Microsoft Excel',
    'Microsoft Word',
    'Microsoft Outlook',
    'Microsoft PowerPoint',
    'Web browser software',
    'Electronic mail software',
}


def _restore_generic_tech(occ: dict, unrefined_occ: dict) -> int:
    """Re-add whitelisted generic tech skills if they were wrongly removed. Returns count restored."""
    refined_names = {ts.get('name', '') for ts in occ.get('technology_skills', [])}
    unrefined_skills = unrefined_occ.get('technology_skills', [])

    restored = 0
    for ts in unrefined_skills:
        name = ts.get('name', '')
        if name in GENERIC_TECH_WHITELIST and name not in refined_names:
            occ.setdefault('technology_skills', []).append(ts)
            refined_names.add(name)
            restored += 1

    return restored










## 1.4 US to UK Terminology Substitution

All task text originates from US O*NET, so it contains US-specific terms
(federal, Medicare, OSHA, etc.). We apply deterministic string substitutions
to replace the most common US terms with UK equivalents. We also remove
technology skills for US-only government systems (FRESA, SEVIS, USDA).

In [ ]:
#|export
# Order matters: longer/more specific patterns first
_US_UK_TASK_SUBS = [
    ('federal and state', 'central and devolved government'),
    ('state and federal', 'central and devolved government'),
    ('Federal and state', 'Central and devolved government'),
    ('State and federal', 'Central and devolved government'),
    ('Federal and State', 'Central and Devolved Government'),
    # "federal" standalone (not part of already-replaced phrase)
    ('federal law', 'national law'),
    ('Federal law', 'National law'),
    ('federal regulation', 'national regulation'),
    ('Federal regulation', 'National regulation'),
    ('federal funding', 'government funding'),
    ('Federal funding', 'Government funding'),
    ('federal requirement', 'national requirement'),
    ('Federal requirement', 'National requirement'),
    ('federal agencies', 'government agencies'),
    ('Federal agencies', 'Government agencies'),
    ('federal ', 'central government '),
    ('Federal ', 'Central government '),
    ('Medicare', 'NHS'),
    ('Medicaid', 'NHS'),
    (' OSHA ', ' HSE '),
]

_US_TECH_REMOVE_PATTERNS = ['FRESA', 'SEVIS', 'USDA']


def _substitute_us_uk_terms(occ: dict) -> int:
    """Apply US->UK term substitutions in tasks and remove US-specific tech. Returns count of changes."""
    changes = 0

    # Tasks: text substitution
    for task in occ.get('tasks', []):
        text = task.get('task', '')
        new_text = text
        for us, uk in _US_UK_TASK_SUBS:
            if us in new_text:
                new_text = new_text.replace(us, uk)
        if new_text != text:
            task['task'] = new_text
            changes += 1

    # Tech skills: remove US-specific entries
    tech = occ.get('technology_skills', [])
    if tech:
        clean = []
        for ts in tech:
            name = ts.get('name', '')
            if any(pat in name for pat in _US_TECH_REMOVE_PATTERNS):
                changes += 1
                continue
            clean.append(ts)
        occ['technology_skills'] = clean

    return changes










## 1.5 Remove Wrong-Domain Tasks

Crosswalk noise sometimes maps occupations from entirely different domains.
The LLM misses some of these because the tasks sound plausible in isolation.
Here we apply targeted keyword rules: gambling-related tasks are removed
from non-gambling occupations, and firefighting tasks from non-fire occupations.

In [ ]:
#|export
_GAMBLING_KEYWORDS = re.compile(
    r'\bgambling\b|\bcasino\b|\bjackpot\b|\bslot machine\b|\bpoker\b|\bgaming table\b|\bwager\b',
    re.IGNORECASE,
)
_GAMBLING_TITLE_WORDS = re.compile(
    r'\bgambling\b|\bbetting\b|\bcasino\b',
    re.IGNORECASE,
)

_FIRE_KEYWORDS = re.compile(
    r'\bfire suppression\b|\barson investigation\b|\bfire crew\b|\bfire station\b|\bfirefight',
    re.IGNORECASE,
)
_FIRE_TITLE_WORDS = re.compile(
    r'\bfire\b|\brescue\b|\bfirefight',
    re.IGNORECASE,
)


def _remove_wrong_domain_tasks(occ: dict) -> int:
    """Remove clearly wrong-domain tasks (gambling in non-gambling, fire in non-fire). Returns count removed."""
    title = occ.get('title', '')
    tasks = occ.get('tasks', [])
    if not tasks:
        return 0

    removed = 0
    clean = []
    for task in tasks:
        text = task.get('task', '')

        # Gambling tasks in non-gambling occupations
        if not _GAMBLING_TITLE_WORDS.search(title) and _GAMBLING_KEYWORDS.search(text):
            removed += 1
            continue

        # Fire tasks in non-fire occupations
        if not _FIRE_TITLE_WORDS.search(title) and _FIRE_KEYWORDS.search(text):
            removed += 1
            continue

        clean.append(task)

    occ['tasks'] = clean
    return removed










## 1.6 Auto-detect Partial Profile Occupations

Some O*NET occupations have only a partial profile (no skills, abilities,
or knowledge data). When a UK SOC code maps exclusively to such occupations,
the resulting profile is empty in these key categories. We flag these so the
frontend can display a notice rather than an empty page.

In [ ]:
#|export
_KEY_CATEGORIES = ('skills', 'abilities', 'knowledge')


def _flag_partial_profiles(occ: dict) -> bool:
    """Flag occupations missing all key data categories (skills, abilities, knowledge).

    These correspond to O*NET occupations with only a partial profile available.
    Returns True if flag was set.
    """
    # Don't overwrite an existing flag (e.g. from manual overrides)
    if occ.get('insufficient_source_data'):
        return False

    missing = [cat for cat in _KEY_CATEGORIES if not occ.get(cat)]
    if len(missing) == len(_KEY_CATEGORIES):
        occ['insufficient_source_data'] = (
            'Source O*NET occupation(s) have only a partial profile — '
            'skills, abilities, and knowledge data are not available'
        )
        return True
    return False










## Manual Overrides

A JSON file (`manual_overrides.json`) allows targeted corrections that are
too specific for general rules. Supported actions include removing tasks by
regex pattern, removing named tech skills, and setting flags on specific
occupations. This is the escape hatch for edge cases that neither the LLM
nor the deterministic rules handle well.

In [ ]:
#|export
import json
from pathlib import Path


def apply_manual_overrides(
    occupations: list[dict],
    overrides_path: Path | None = None,
) -> list[dict]:
    """
    Apply manual overrides from a JSON file as the final pipeline step.

    Supported actions:
    - remove_tasks_containing: remove tasks matching a regex pattern
    - remove_tech_skills: remove tech skills by exact name
    - set_flag: add a flag/reason to the occupation dict

    Args:
        occupations: List of occupation dicts (mutated in place).
        overrides_path: Path to manual_overrides.json.

    Returns:
        The occupation list with overrides applied.
    """
    if overrides_path is None:
        overrides_path = Path(__file__).parent.parent / 'manual_overrides.json'
    if not overrides_path.exists():
        return occupations

    with open(overrides_path) as f:
        data = json.load(f)

    overrides = data.get('overrides', [])
    if not overrides:
        return occupations

    # Build lookup
    occ_lookup = {occ['uk_soc_2020']: occ for occ in occupations}
    applied = 0

    for override in overrides:
        code = override['uk_soc_2020']
        action = override['action']
        occ = occ_lookup.get(code)
        if not occ:
            continue

        if action == 'remove_tasks_containing':
            pattern = re.compile(override['pattern'], re.IGNORECASE)
            before = len(occ.get('tasks', []))
            occ['tasks'] = [t for t in occ.get('tasks', []) if not pattern.search(t.get('task', ''))]
            removed = before - len(occ['tasks'])
            if removed:
                logger.info(f"SOC {code}: removed {removed} tasks matching '{override['pattern']}'")
                applied += removed

        elif action == 'remove_tech_skills':
            names_to_remove = set(override['names'])
            before = len(occ.get('technology_skills', []))
            occ['technology_skills'] = [
                ts for ts in occ.get('technology_skills', [])
                if ts.get('name', '') not in names_to_remove
            ]
            removed = before - len(occ['technology_skills'])
            if removed:
                logger.info(f"SOC {code}: removed {removed} tech skills")
                applied += removed

        elif action == 'set_flag':
            occ[override['flag']] = override.get('reason', True)
            applied += 1

    if applied:
        print(f"Manual overrides: {applied} changes applied from {overrides_path.name}")

    return occupations










## Dataset-Level Entry Point

Runs all post-processing steps over the full dataset in order. Steps are
applied sequentially per occupation: normalise task types, remove placeholders,
restore whitelisted tech, substitute US→UK terms, remove wrong-domain tasks,
and flag partial profiles. Prints a summary of all changes made.

In [ ]:
#|export
def postprocess_dataset(
    occupations: list[dict],
    unrefined_occupations: list[dict] | None = None,
) -> list[dict]:
    """
    Apply deterministic post-processing fixes to refined occupation data.

    Args:
        occupations: List of refined occupation dicts (mutated in place).
        unrefined_occupations: Optional list of unrefined occupation dicts
            (needed for tech skill whitelist restoration).

    Returns:
        The post-processed list of occupation dicts.
    """
    # Build lookup for unrefined data
    unrefined_lookup = {}
    if unrefined_occupations:
        for occ in unrefined_occupations:
            unrefined_lookup[occ['uk_soc_2020']] = occ

    total_type_fixes = 0
    total_placeholders = 0
    total_tech_restored = 0
    total_us_uk = 0
    total_wrong_domain = 0
    total_partial = 0

    for occ in occupations:
        code = occ.get('uk_soc_2020', 0)

        # 1.1 Normalise task_types
        total_type_fixes += _normalise_task_types(occ)

        # 1.2 Remove placeholder tasks
        total_placeholders += _remove_placeholder_tasks(occ)

        # 1.3 Restore generic tech skills
        unrefined = unrefined_lookup.get(code)
        if unrefined:
            total_tech_restored += _restore_generic_tech(occ, unrefined)

        # 1.4 US->UK terminology
        total_us_uk += _substitute_us_uk_terms(occ)

        # 1.5 Wrong-domain tasks
        total_wrong_domain += _remove_wrong_domain_tasks(occ)

        # 1.6 Flag partial profiles
        if _flag_partial_profiles(occ):
            total_partial += 1

    print(
        f"Post-processing complete: "
        f"{total_type_fixes} task_types normalised, "
        f"{total_placeholders} placeholder tasks removed, "
        f"{total_tech_restored} generic tech skills restored, "
        f"{total_us_uk} US->UK substitutions, "
        f"{total_wrong_domain} wrong-domain tasks removed, "
        f"{total_partial} partial profiles flagged"
    )

    return occupations